# Commodity News Sentiment Pipeline with DeBERTa

This notebook is a cleaner end-to-end pipeline for commodity news sentiment analysis.

It does four things in a simple order:

1. Load and clean the raw news dataset  
2. Score each headline with a DeBERTa sentiment model  
3. Split headlines into **oil**, **gas**, **silver**, and **gold**  
4. Aggregate sentiment by **date + topic** and export CSV files

## Expected input
The raw news file should contain at least:
- a headline column such as `title`, `headline`, or `news_title`
- a date column such as `published_at`, `date`, `published`, or `datetime`

## Outputs
This notebook writes two CSV files:
- `news_with_topics_and_sentiment.csv` → one row per headline
- `daily_topic_sentiment.csv` → one row per date per topic

In [1]:

# =========================
# 0. Imports
# =========================
from pathlib import Path
import json
import re
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tqdm.pandas()
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 160)

## 1. Configuration

Edit only the file paths below if your input file lives somewhere else.

In [9]:

# =========================
# 1. Configuration
# =========================
BASE_DIR = Path.cwd()
PARENT_DIR = BASE_DIR.parent
DATA_DIR = PARENT_DIR / "data"
NEWS_DIR = DATA_DIR / "raw" / "news" / "2y_daily"
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Put your raw news file here.
# Supported formats: .csv, .json, .jsonl
NEWS_INPUT_PATH = NEWS_DIR / "bt_energy_commodities_2y_5k.jsonl"

# Output files
NEWS_SCORED_OUTPUT_PATH = OUTPUT_DIR / "news_with_topics_and_sentiment.csv"
DAILY_TOPIC_OUTPUT_PATH = OUTPUT_DIR / "daily_topic_sentiment.csv"

# DeBERTa sentiment checkpoint
MODEL_NAME = "mrm8488/deberta-v3-ft-financial-news-sentiment-analysis"

# Maximum token length for each headline
MAX_LENGTH = 192
BATCH_SIZE = 32

In [10]:
## debugging
from pathlib import Path
import os

print("cwd:", Path.cwd())
print("files here:", os.listdir())

cwd: c:\Users\diant\OneDrive\Documents\Term 8\CDS\DS_Project_g8\finetuning
files here: ['01_feature_engineering_experiments.ipynb', '02_tree_model_tuning.ipynb', '03_lstm_redesign.ipynb', '04_deberta_sentiment_analysis.ipynb', 'data', 'notebook.py', 'outputs', '__pycache__']


## 2. Helper functions

These functions keep the notebook tidy and make each step easier to debug.

In [11]:

# =========================
# 2. Helper functions
# =========================
def load_structured_file(path: Path) -> pd.DataFrame:
    """Load CSV, JSON, JSONL, or newline-delimited JSON into a pandas DataFrame."""
    if not path.exists():
        raise FileNotFoundError(f"Input file not found: {path}")

    suffix = path.suffix.lower()

    if suffix == ".csv":
        return pd.read_csv(path)

    if suffix in {".json", ".jsonl", ".txt"}:
        text = path.read_text(encoding="utf-8").strip()

        if not text:
            raise ValueError(f"Input file is empty: {path}")

        # First try standard JSON parsing
        try:
            obj = json.loads(text)
            if isinstance(obj, list):
                return pd.DataFrame(obj)
            if isinstance(obj, dict):
                if "data" in obj and isinstance(obj["data"], list):
                    return pd.DataFrame(obj["data"])
                return pd.DataFrame([obj])
        except json.JSONDecodeError:
            pass

        # Fallback: newline-delimited JSON
        records = []
        for line_no, line in enumerate(text.splitlines(), start=1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise ValueError(
                    f"Could not parse line {line_no} as JSON in {path}: {e}"
                ) from e

        if records:
            return pd.DataFrame(records)

    raise ValueError(f"Unsupported or unreadable file type: {suffix}")


def standardize_news_columns(df_raw: pd.DataFrame) -> pd.DataFrame:
    """Map likely news columns into a standard schema for this pipeline."""
    df = df_raw.copy()

    headline_candidates = [
        "headline", "title", "news_title", "article_title", "subject"
    ]
    date_candidates = [
        "date", "published_at", "published", "datetime", "timestamp", "created_at"
    ]
    summary_candidates = ["summary", "description", "content", "snippet"]
    url_candidates = ["url", "link", "article_url", "source_url"]
    source_candidates = ["source", "publisher", "site", "domain", "section"]

    def first_existing(candidates):
        for col in candidates:
            if col in df.columns:
                return col
        return None

    headline_col = first_existing(headline_candidates)
    date_col = first_existing(date_candidates)
    summary_col = first_existing(summary_candidates)
    url_col = first_existing(url_candidates)
    source_col = first_existing(source_candidates)

    if headline_col is None:
        raise KeyError(
            "Could not find a headline/title column. Expected one of: "
            f"{headline_candidates}"
        )

    if date_col is None:
        raise KeyError(
            "Could not find a date column. Expected one of: "
            f"{date_candidates}"
        )

    renamed = df.rename(columns={
        headline_col: "headline",
        date_col: "date"
    }).copy()

    keep_cols = ["headline", "date"]
    if summary_col:
        renamed = renamed.rename(columns={summary_col: "summary"})
        keep_cols.append("summary")
    if url_col:
        renamed = renamed.rename(columns={url_col: "url"})
        keep_cols.append("url")
    if source_col:
        renamed = renamed.rename(columns={source_col: "source"})
        keep_cols.append("source")

    renamed = renamed[keep_cols].copy()

    renamed["headline"] = renamed["headline"].fillna("").astype(str).str.strip()
    renamed["date"] = pd.to_datetime(renamed["date"], errors="coerce").dt.date

    if "summary" not in renamed.columns:
        renamed["summary"] = ""

    renamed["summary"] = renamed["summary"].fillna("").astype(str).str.strip()

    for col in ["headline", "summary"]:
        renamed[col] = (
            renamed[col]
            .astype(str)
            .str.replace("â€™", "'", regex=False)
            .str.replace("â€œ", '"', regex=False)
            .str.replace("â€\x9d", '"', regex=False)
            .str.replace("â€“", "-", regex=False)
            .str.strip()
        )

    # Use title + summary together for topic detection and sentiment context
    renamed["text_for_model"] = (
        renamed["headline"].fillna("").astype(str).str.strip()
        + ". "
        + renamed["summary"].fillna("").astype(str).str.strip()
    ).str.strip()

    # Remove accidental leading punctuation if summary is empty
    renamed["text_for_model"] = (
        renamed["text_for_model"]
        .str.replace(r"^\.\s*", "", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    renamed = renamed.dropna(subset=["headline", "date"])
    renamed = renamed[renamed["headline"] != ""].reset_index(drop=True)

    return renamed


TOPIC_PATTERNS = {
    "oil": re.compile(
        r"\b(oil|crude|brent|wti|petroleum|diesel|gasoline|refinery|opec|opec\+)\b",
        flags=re.IGNORECASE,
    ),
    "gas": re.compile(
        r"\b(natural gas|natgas|lng|pipeline gas|gas storage|gas supply|city energy|ttf|henry hub)\b",
        flags=re.IGNORECASE,
    ),
    "gold": re.compile(
        r"\b(gold|bullion|safe[- ]haven metal)\b",
        flags=re.IGNORECASE,
    ),
    "silver": re.compile(
        r"\b(silver|xag|precious silver)\b",
        flags=re.IGNORECASE,
    ),
}


def assign_topic(text: str) -> str:
    """Assign a single topic label based on regex rules."""
    text = str(text)

    matched_topics = [
        topic for topic, pattern in TOPIC_PATTERNS.items()
        if pattern.search(text)
    ]

    if len(matched_topics) == 0:
        return "other"

    priority = ["oil", "gas", "gold", "silver"]
    for topic in priority:
        if topic in matched_topics:
            return topic

    return matched_topics[0]


def score_batch_texts(texts, tokenizer, model, device, max_length=128):
    """Run batch sentiment inference and return probability array."""
    enc = tokenizer(
        list(map(str, texts)),
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=max_length
    )
    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        outputs = model(**enc)

    probs = F.softmax(outputs.logits, dim=-1).cpu().numpy()
    return probs


def extract_sentiment_scores(prob_row, id2label):
    """Convert model probabilities into positive/negative/neutral scores."""
    label_to_prob = {}
    for idx, prob in enumerate(prob_row):
        raw_label = str(id2label.get(idx, f"label_{idx}")).lower().strip()
        label_to_prob[raw_label] = float(prob)

    positive = next((v for k, v in label_to_prob.items() if "pos" in k), 0.0)
    negative = next((v for k, v in label_to_prob.items() if "neg" in k), 0.0)
    neutral = next((v for k, v in label_to_prob.items() if "neu" in k), 0.0)

    sentiment_score = (positive - negative) * (1 - neutral)

    return {
        "prob_positive": positive,
        "prob_negative": negative,
        "prob_neutral": neutral,
        "sentiment_score": float(sentiment_score),
    }


def score_headlines_dataframe(df, tokenizer, model, device, batch_size=32, max_length=128):
    """Score all articles and return a dataframe with sentiment columns."""
    rows = []
    text_col = "text_for_model" if "text_for_model" in df.columns else "headline"

    for start in tqdm(range(0, len(df), batch_size), desc="Scoring articles"):
        batch = df.iloc[start:start + batch_size].copy()
        probs = score_batch_texts(
            batch[text_col].tolist(),
            tokenizer=tokenizer,
            model=model,
            device=device,
            max_length=max_length
        )

        for i, (_, row) in enumerate(batch.iterrows()):
            sent = extract_sentiment_scores(probs[i], model.config.id2label)
            rows.append({
                **row.to_dict(),
                **sent
            })

    return pd.DataFrame(rows)


def aggregate_daily_topic_sentiment(df_scored: pd.DataFrame) -> pd.DataFrame:
    """Aggregate sentiment by date and topic."""
    daily = (
        df_scored[df_scored["topic"].isin(["oil", "gas", "gold", "silver"])]
        .groupby(["date", "topic"], as_index=False)
        .agg(
            news_volume=("headline", "count"),
            avg_sentiment=("sentiment_score", "mean"),
            median_sentiment=("sentiment_score", "median"),
            sentiment_std=("sentiment_score", "std"),
            max_sentiment=("sentiment_score", "max"),
            min_sentiment=("sentiment_score", "min"),
            avg_prob_positive=("prob_positive", "mean"),
            avg_prob_negative=("prob_negative", "mean"),
            avg_prob_neutral=("prob_neutral", "mean"),
        )
        .sort_values(["date", "topic"])
        .reset_index(drop=True)
    )

    daily["sentiment_std"] = daily["sentiment_std"].fillna(0.0)
    daily["sentiment_volume_weighted"] = daily["avg_sentiment"] * np.log1p(daily["news_volume"])

    return daily


## 3. Load and inspect the raw news data

In [12]:

# =========================
# 3. Load the raw news data
# =========================
df_raw = load_structured_file(NEWS_INPUT_PATH)

print("Raw shape:", df_raw.shape)
display(df_raw.head())
print("Columns:", list(df_raw.columns))

Raw shape: (5118, 5)


,title,summary,url,published_at,section
0,"WTI, June Brent crude futures settle down on reports that Iran may be ready to end war",[HOUSTON] Brent futures for June delivery settled down more than US$3 on Tuesday following unconfirmed media reports that Iran’s president said the country ...,https://www.businesstimes.com.sg/companies-markets/energy-commodities/wti-june-brent-crude-futures-settle-down-reports-iran-may-be-ready-end-war,2026-03-31T22:16:06+00:00,Energy & Commodities
1,‘Real pain’ to set in: Businesses scramble to cope as EMA warns of sharper rise in power tariffs,Utility companies SP Group and City Energy announce price hikes for Q2 2026,https://www.businesstimes.com.sg/companies-markets/energy-commodities/real-pain-set-businesses-scramble-cope-ema-warns-sharper-rise-power-tariffs,2026-03-31T15:05:32+00:00,Energy & Commodities
2,China set to extend fuel export ban with small exemptions: sources,"Countries that may receive Chinese fuel supplies include Bangladesh, Myanmar, Sri Lanka, the Maldives and Vietnam",https://www.businesstimes.com.sg/international/global/china-set-extend-fuel-export-ban-small-exemptions-sources,2026-03-31T14:33:30+00:00,Energy & Commodities
3,Singapore chemical logistics growth still promising despite Middle East war: Katoen Natie execs,"Specialty chemical producers are scaling up their plants on Jurong Island, driving demand for warehouse storage",https://www.businesstimes.com.sg/companies-markets/energy-commodities/singapore-chemical-logistics-growth-still-promising-despite-middle-east-war-katoen-nat...,2026-03-31T12:52:25+00:00,Energy & Commodities
4,Eurozone inflation jumps most since 2022 on energy costs,Expectations are that the ECB will have to raise interest rates,https://www.businesstimes.com.sg/international/eurozone-inflation-jumps-most-2022-energy-costs,2026-03-31T12:44:37+00:00,Energy & Commodities


Columns: ['title', 'summary', 'url', 'published_at', 'section']


## 4. Clean and standardize the news data

In [13]:

# =========================
# 4. Clean and standardize
# =========================
df_news = standardize_news_columns(df_raw)

print("Clean shape:", df_news.shape)
display(df_news.head())
print("Date range:", df_news["date"].min(), "to", df_news["date"].max())

Clean shape: (5118, 6)


,headline,date,summary,url,source,text_for_model
0,"WTI, June Brent crude futures settle down on reports that Iran may be ready to end war",2026-03-31,[HOUSTON] Brent futures for June delivery settled down more than US$3 on Tuesday following unconfirmed media reports that Iran’s president said the country ...,https://www.businesstimes.com.sg/companies-markets/energy-commodities/wti-june-brent-crude-futures-settle-down-reports-iran-may-be-ready-end-war,Energy & Commodities,"WTI, June Brent crude futures settle down on reports that Iran may be ready to end war. [HOUSTON] Brent futures for June delivery settled down more than US$..."
1,‘Real pain’ to set in: Businesses scramble to cope as EMA warns of sharper rise in power tariffs,2026-03-31,Utility companies SP Group and City Energy announce price hikes for Q2 2026,https://www.businesstimes.com.sg/companies-markets/energy-commodities/real-pain-set-businesses-scramble-cope-ema-warns-sharper-rise-power-tariffs,Energy & Commodities,‘Real pain’ to set in: Businesses scramble to cope as EMA warns of sharper rise in power tariffs. Utility companies SP Group and City Energy announce price ...
2,China set to extend fuel export ban with small exemptions: sources,2026-03-31,"Countries that may receive Chinese fuel supplies include Bangladesh, Myanmar, Sri Lanka, the Maldives and Vietnam",https://www.businesstimes.com.sg/international/global/china-set-extend-fuel-export-ban-small-exemptions-sources,Energy & Commodities,"China set to extend fuel export ban with small exemptions: sources. Countries that may receive Chinese fuel supplies include Bangladesh, Myanmar, Sri Lanka,..."
3,Singapore chemical logistics growth still promising despite Middle East war: Katoen Natie execs,2026-03-31,"Specialty chemical producers are scaling up their plants on Jurong Island, driving demand for warehouse storage",https://www.businesstimes.com.sg/companies-markets/energy-commodities/singapore-chemical-logistics-growth-still-promising-despite-middle-east-war-katoen-nat...,Energy & Commodities,Singapore chemical logistics growth still promising despite Middle East war: Katoen Natie execs. Specialty chemical producers are scaling up their plants on...
4,Eurozone inflation jumps most since 2022 on energy costs,2026-03-31,Expectations are that the ECB will have to raise interest rates,https://www.businesstimes.com.sg/international/eurozone-inflation-jumps-most-2022-energy-costs,Energy & Commodities,Eurozone inflation jumps most since 2022 on energy costs. Expectations are that the ECB will have to raise interest rates


Date range: 2024-04-01 to 2026-03-31


## 5. Load the DeBERTa sentiment model

In [14]:

# =========================
# 5. Load DeBERTa
# =========================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

print("Device:", device)
print("Label mapping:", model.config.id2label)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/568M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

Device: cuda
Label mapping: {0: 'negative', 1: 'neutral', 2: 'positive'}


## 6. Score each headline with DeBERTa

In [15]:

# =========================
# 6. Score sentiment
# =========================
df_scored = score_headlines_dataframe(
    df_news,
    tokenizer=tokenizer,
    model=model,
    device=device,
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH
)

display(
    df_scored[
        ["date", "headline", "prob_positive", "prob_negative", "prob_neutral", "sentiment_score"]
    ].head(10)
)

df_scored["sentiment_score"].describe()

Scoring articles:   0%|          | 0/160 [00:00<?, ?it/s]

,date,headline,prob_positive,prob_negative,prob_neutral,sentiment_score
0,2026-03-31,"WTI, June Brent crude futures settle down on reports that Iran may be ready to end war",0.018353,0.659601,0.322046,-0.434737
1,2026-03-31,‘Real pain’ to set in: Businesses scramble to cope as EMA warns of sharper rise in power tariffs,0.001006,0.994209,0.004785,-0.988450
2,2026-03-31,China set to extend fuel export ban with small exemptions: sources,0.005397,0.002833,0.991771,0.000021
3,2026-03-31,Singapore chemical logistics growth still promising despite Middle East war: Katoen Natie execs,0.998663,0.000294,0.001043,0.997328
4,2026-03-31,Eurozone inflation jumps most since 2022 on energy costs,0.996120,0.000764,0.003117,0.992254
5,2026-03-31,India diesel exports to South-east Asia hit seven-year high in March due to Iran war,0.994973,0.001402,0.003625,0.989970
6,2026-03-31,A forgotten crisis explains today’s oil shock,0.028320,0.740400,0.231281,-0.547389
7,2026-03-31,Malaysia’s Petronas Chemicals tops Asia stocks with 102% rally,0.997546,0.000425,0.002028,0.995099
8,2026-03-31,Indonesia's Pertamina will not increase fuel prices on April 1,0.000933,0.006817,0.992250,-0.000046
9,2026-03-31,Malaysia says its tankers will be exempt from Iran’s Hormuz toll,0.013226,0.001103,0.985671,0.000174


count    5118.000000
mean        0.049729
std         0.769455
min        -0.998052
25%        -0.876207
50%         0.000115
75%         0.955852
max         0.998233
Name: sentiment_score, dtype: float64

## 7. Split each headline into a commodity topic

This version uses transparent regex rules so the topic assignment is easier to debug and explain.

In [16]:

# =========================
# 7. Assign topic
# =========================
df_scored["topic"] = df_scored["headline"].apply(assign_topic)

print(df_scored["topic"].value_counts(dropna=False))
display(df_scored[["date", "headline", "topic", "sentiment_score"]].head(20))

topic
other     2795
oil       1325
gold       866
gas         86
silver      46
Name: count, dtype: int64


,date,headline,topic,sentiment_score
0,2026-03-31,"WTI, June Brent crude futures settle down on reports that Iran may be ready to end war",oil,-0.434737
1,2026-03-31,‘Real pain’ to set in: Businesses scramble to cope as EMA warns of sharper rise in power tariffs,other,-0.988450
2,2026-03-31,China set to extend fuel export ban with small exemptions: sources,other,0.000021
3,2026-03-31,Singapore chemical logistics growth still promising despite Middle East war: Katoen Natie execs,other,0.997328
4,2026-03-31,Eurozone inflation jumps most since 2022 on energy costs,other,0.992254
5,2026-03-31,India diesel exports to South-east Asia hit seven-year high in March due to Iran war,oil,0.989970
6,2026-03-31,A forgotten crisis explains today’s oil shock,oil,-0.547389
7,2026-03-31,Malaysia’s Petronas Chemicals tops Asia stocks with 102% rally,other,0.995099
8,2026-03-31,Indonesia's Pertamina will not increase fuel prices on April 1,other,-0.000046
9,2026-03-31,Malaysia says its tankers will be exempt from Iran’s Hormuz toll,other,0.000174


## 8. Keep the four target topics only

In [17]:

# =========================
# 8. Filter target topics
# =========================
target_topics = ["oil", "gas", "gold", "silver"]
df_target = df_scored[df_scored["topic"].isin(target_topics)].copy()

print("Rows kept:", len(df_target))
display(df_target[["date", "headline", "topic", "sentiment_score"]].head(20))

Rows kept: 2323


,date,headline,topic,sentiment_score
0,2026-03-31,"WTI, June Brent crude futures settle down on reports that Iran may be ready to end war",oil,-0.434737
5,2026-03-31,India diesel exports to South-east Asia hit seven-year high in March due to Iran war,oil,0.989970
6,2026-03-31,A forgotten crisis explains today’s oil shock,oil,-0.547389
12,2026-03-31,Oil could spike to US$200 if Hormuz remains shut: energy consultancy,oil,-0.039292
16,2026-03-31,Gold set for worst month in more than 17 years as US rate-cut hopes fade,gold,-0.996384
19,2026-03-30,Brent eyes record monthly rise; US crude settles above US$100 as Houthis join Iran war,oil,0.908181
23,2026-03-30,Philippines’ businesses hoped to turn the corner in 2026 – then oil prices spiked overnight,oil,-0.303577
25,2026-03-30,Iran war forces Asian economies to confront sliding currencies and surging oil,oil,-0.996197
26,2026-03-30,"Singapore investors buy the dip, bucking global gold sell-off: OCBC, bullion dealers",gold,0.736273
27,2026-03-30,Gold steady as softer dollar offsets fading Fed rate-cut hopes,gold,-0.962547


## 9. Aggregate by date and topic

In [18]:

# =========================
# 9. Aggregate by date and topic
# =========================
daily_topic_sentiment = aggregate_daily_topic_sentiment(df_target)

print("Daily-topic shape:", daily_topic_sentiment.shape)
display(daily_topic_sentiment.head(20))

Daily-topic shape: (1266, 12)


,date,topic,news_volume,avg_sentiment,median_sentiment,sentiment_std,max_sentiment,min_sentiment,avg_prob_positive,avg_prob_negative,avg_prob_neutral,sentiment_volume_weighted
0,2024-04-01,oil,1,0.988299,0.988299,0.000000,0.988299,0.988299,0.994134,0.001945,0.003921,0.685037
1,2024-04-02,gold,1,0.968506,0.968506,0.000000,0.968506,0.968506,0.984128,0.001370,0.014502,0.671317
2,2024-04-02,oil,1,0.993794,0.993794,0.000000,0.993794,0.993794,0.996893,0.000932,0.002175,0.688846
3,2024-04-03,gold,2,0.493880,0.493880,0.698429,0.987744,0.000016,0.498986,0.001235,0.499778,0.542583
4,2024-04-03,oil,2,-0.016079,-0.016079,1.222358,0.848259,-0.880416,0.461802,0.482097,0.056100,-0.017664
5,2024-04-04,gold,1,0.990232,0.990232,0.000000,0.990232,0.990232,0.995105,0.001362,0.003533,0.686377
6,2024-04-04,oil,3,0.329663,0.991329,1.149543,0.995373,-0.997713,0.664546,0.333422,0.002033,0.457010
7,2024-04-05,gold,2,-0.889725,-0.889725,0.042363,-0.859769,-0.919680,0.015017,0.943261,0.041722,-0.977462
8,2024-04-05,oil,3,0.362487,0.974451,1.072164,0.988525,-0.875514,0.664835,0.313271,0.021894,0.502514
9,2024-04-07,gold,1,0.000018,0.000018,0.000000,0.000018,0.000018,0.004193,0.000161,0.995646,0.000012


## 10. Export the CSV files

This writes both:
- per-headline sentiment results
- daily aggregated topic sentiment

In [19]:

# =========================
# 10. Save outputs
# =========================
df_target.to_csv(NEWS_SCORED_OUTPUT_PATH, index=False)
daily_topic_sentiment.to_csv(DAILY_TOPIC_OUTPUT_PATH, index=False)

print("Saved:", NEWS_SCORED_OUTPUT_PATH)
print("Saved:", DAILY_TOPIC_OUTPUT_PATH)

Saved: c:\Users\diant\OneDrive\Documents\Term 8\CDS\DS_Project_g8\finetuning\outputs\news_with_topics_and_sentiment.csv
Saved: c:\Users\diant\OneDrive\Documents\Term 8\CDS\DS_Project_g8\finetuning\outputs\daily_topic_sentiment.csv


## 11. Optional wide-format table

This is useful if you later want one row per date and separate sentiment columns for each topic.

In [20]:

# =========================
# 11. Optional wide-format pivot
# =========================
wide_daily = (
    daily_topic_sentiment
    .pivot(index="date", columns="topic", values="avg_sentiment")
    .reset_index()
    .rename_axis(None, axis=1)
)

display(wide_daily.head(20))

,date,gas,gold,oil,silver
0,2024-04-01,NaN,NaN,0.988299,NaN
1,2024-04-02,NaN,0.968506,0.993794,NaN
2,2024-04-03,NaN,0.493880,-0.016079,NaN
3,2024-04-04,NaN,0.990232,0.329663,NaN
4,2024-04-05,NaN,-0.889725,0.362487,NaN
5,2024-04-07,NaN,0.000018,0.982710,NaN
6,2024-04-08,NaN,0.416127,0.015399,NaN
7,2024-04-09,NaN,0.986974,-0.530932,NaN
8,2024-04-10,NaN,0.965567,0.993208,NaN
9,2024-04-11,NaN,0.816559,-0.008161,NaN
